# PROJECT_ASSAY_NAME - YYYY-MM-DD - LC-MS XCMS

This document provides a pipeline for the import of LC-MS XCMS data, and any associated sample metadata, followed by summaries and quality control reports of the data, implementation of pre-processing and quality control analysis and output of a final dataset ready for sharing with collaborators and data modeling.

# 1. Initial Setup

### Define File Paths

NOTE: file paths provided here are relative, with the understanding that this notebook is run from inside the standard NPC project folder structure:

/Study Data/Metadata/nPYc pipeline notebooks

If the notebook is not run from this location, full file paths must be provided.

Preceding the file path (in quotes) with the letter r ensures that the paths are interpreted exactly as written.

In [ ]:
# Project assay specific name (in the standard NPC format)
datasetName = 'PROJECT_LCMS_BIOFLUID_ASSAY'

# Path to peak-picked (XCMS) csv data file
csvDataPath = r'../../Analytical/Preprocessed/PROJECT_BIOFLUID_ASSAY/csvFileName.csv'

# Path to raw data
rawDataPath = r'../../Analytical/Raw/RAW_DATA.PRO/Data'

# Path to NPC data locations file (downloaded from NPC LIMS)
limsFilePath = r'../Data locations/Project_data_locations.csv'

# Path to NPC sample manifest file (provided by project team)
manifestPath = r'../LIMS files/NPC_Project_SampleManifest.xlsx'

# Path to /npc-admin/live/NPC_code/phenome-utils/npc_template_notebooks
adminPath = r'../../../../../npc-admin/live/NPC_code/phenome-utils/npc_template_notebooks'

# IF REQUIRED: Path to Basic CSV file (generalised alternative to data locations/sample manifest)
#basiccsvPath = r'../Data locations/Project_basic_csv.csv'

# IF REQUIRED: Path to reduced dilution series file linking sample name to correct dilution factor
dilutionMapPath = adminPath + '/ReducedDilutionSeries.csv'

## Import the nPYc-Toolbox and Configure the Jupyter Notebook

In [ ]:
### CODE FOR DEVELOPMENT:

toolboxPath = '/Users/cs401/workspace/nPYc-Toolbox'

import sys
sys.path.append(toolboxPath)

In [ ]:
# Import the nPYc-Toolbox
import nPYc

# Import enumerations for sample type
from nPYc.enumerations import VariableType, DatasetLevel, AssayRole, SampleType, CalibrationMethod, QuantificationType
# Import normalisation objects for data normalisation
from nPYc.utilities.normalisation import NullNormaliser, TotalAreaNormaliser, ProbabilisticQuotientNormaliser

# Import matplotlib plotting, configure the Jupyter notebook to plot inline
import matplotlib.pyplot as plt
%matplotlib inline

# Set up plotly to work in offline mode with the notebook
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

import plotly.io as pio
pio.renderers.default='notebook'

# Import other required libraries
import os
import numpy
import pandas
from datetime import datetime
import copy

# Set up to hide warnings (particularly Depreciation, RunTime warnings, these are not for the user to worry about!)
# These lines can be commented to show warnings if of interest to advanced users
import warnings
warnings.filterwarnings(action='ignore')

# Print version
from nPYc.__init__ import __version__ as version
print('Run with branch ' + version + ' on ' + datetime.now().strftime("%Y-%m-%d %H:%M"))

# Turn autosave off
%autosave 0

In [ ]:
### EXPORT PATHS FOR DEVELOPMENT

# QC directory
qcDir = '../QC/' + datasetName + '/Profiling_' + datetime.today().strftime('%Y-%m-%d %H-%M-%S') 

# QC directory for uncorrected (batch and run-order correction not applied) data
qcDirUncorrected = os.path.join(qcDir,'Uncorrected')

# QC directory for corrected (batch and run-order correction applied) data
qcDirCorrected = os.path.join(qcDir, 'Corrected')

# Final dataset directory
saveDir = '../Datasets/'+ datasetName + '_Profiling_' + datetime.today().strftime('%Y-%m-%d')

print('qcDir: ' + qcDir)
print('saveDir: ' + saveDir)

# 2. Import Data and Sample Metadata

Import acquired data and associated acqusition parameters:

In [ ]:
msData = nPYc.MSDataset(csvDataPath,
                        fileType='XCMS',
                        noFeatureParams=9)

noFeatureParams can vary dependending on XCMS settings, check in Sample Summary report that all samples are captured, and amend noFeatureParams if required.

In [ ]:
msData.sampleMetadata['Sample File Name']

Owing to the XCMS algorithm occasionally the resulting csv file can contain *Nan* values, these are replaced by zeros using the following line:

In [ ]:
msData._intensityData[numpy.isnan(msData._intensityData)] = 0

In [ ]:
msData.name = datasetName

# 3. Import Sample and Feature Metadata and Match to Acquired Data

The next step is to add sample and feature associated metadata to the Dataset object, and match to the acquired data.

Sample metadata can be added from a variety of sources.

NPC default, infer information from filenames (if standard conventions used) and import information from raw data, data locations and sample manifest files and match to acquired samples.

If samples were acquired with reduced dilution series, before adding information by 'Filenames' the dilution map should be amended using:

```
dilutionMap = pandas.read_csv(dilutionMapPath,
                           index_col='Sample Name')

msData.Attributes['dilutionMap'] = dilutionMap['Dilution Factor (%)'].to_dict()
```

In [ ]:
msData.addSampleInfo(descriptionFormat='Filenames')

In [ ]:
msData.addSampleInfo(descriptionFormat='Raw Data',
                     filePath=rawDataPath)

In [ ]:
msData.addSampleInfo(descriptionFormat='NPC LIMS',
                     filePath=limsFilePath)

In [ ]:
msData.addSampleInfo(descriptionFormat='Infer Batches')

In [ ]:
msData.addSampleInfo(descriptionFormat='NPC Subject Info',
                     filePath=manifestPath)

Alternatively, if non NPC standard import basic csv file and match to acquired samples:

```
msData.addSampleInfo(descriptionFormat='Basic CSV', filePath=basiccsvPath)
```

To add additional analytical parameters available in the sampleMetadata for checking in the multivariate report:

```
msData.Attributes['analyticalMeasurements']['Column Name'] = 'categorical'
msData.Attributes['analyticalMeasurements']['Column Name'] = 'continuous'
```

# 4. Exclude Features with RT Outside of Useful Bounds

Features with retention times outside the useful bounds for each assay should be excluded using (bounds as follows):

- BANEG: 1-12
- HNEG: 0.65-8.0
- HPOS: 0.6-7.5
- LNEG: 0.8-9.5
- LPOS: 0.8-12
- RPOS/RNEG: 0.6-10.5

In [ ]:
rtMin = 0.6
rtMax = 7.5

In [ ]:
print(sum(msData.featureMask))

msData.excludeFeatures(msData.featureMetadata[msData.featureMetadata['Retention Time'] > rtMax]['Feature Name'],
                       on='Feature Name',
                       message='Outside RT limits')

msData.excludeFeatures(msData.featureMetadata[msData.featureMetadata['Retention Time'] < rtMin]['Feature Name'], 
                       on='Feature Name', 
                       message='Outside RT limits')

print(sum(msData.featureMask))

msData.applyMasks()

# 5. Sample & Feature Summary Reports

The nPYc-Toolbox offers a series of reports, pre-set visualisations comprised of text, figures and tables to describe and summarise the characteristics of the dataset, and help the user assess the overall impact of quality control decisions (i.e. excluding samples or features and changing filtering criteria).

### Sample Summary Report

Check the expected samples against those acquired, in terms of numbers, sample type, and any samples either missing from acquisition or not recorded in the LIMS or sample manifest files:

In [ ]:
nPYc.reports.generateReport(msData, 'sample summary')

Exclude any samples marked for exclusion ('_X') by data analyst (these are typically missing/low volume):

In [ ]:
msData.excludeSamples(msData.sampleMetadata.iloc[msData.sampleMetadata['Skipped'].values==True]['Sample File Name'],
                      on='Sample File Name',
                      message='Missing/low volume')

To help aid decisions on samples which should be removed an interactive plot of the sum of feature intensities for all samples can be generated using:

In [ ]:
data = nPYc.plotting.plotIntensityInteractive(msData,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True)
iplot(data)

Exclude any analytically compromised samples:

In [ ]:
# msData.excludeSamples(['Sample File Name to be excluded'],
#                       on='Sample File Name',
#                       message='Compromised sample')

Exclude any batch termini samples (typical naming convention B1S1-5, B1E1-5):

In [ ]:
mask = msData.sampleMetadata['Sample File Name'].str.match('.+[B]\d+?[SE]\d+?', na=False).astype(bool)

msData.sampleMetadata.loc[mask, 'Sample File Name']

In [ ]:
msData.excludeSamples(msData.sampleMetadata.loc[mask, 'Sample File Name'],
                      on='Sample File Name',
                      message='Batch termini SR')

Exclude any blank samples:

In [ ]:
mask = msData.sampleMetadata['Sample File Name'].str.match('.+Blank\d+?', na=False).astype(bool)

msData.sampleMetadata.loc[mask, 'Sample File Name']

In [ ]:
msData.excludeSamples(msData.sampleMetadata.loc[mask, 'Sample File Name'],
                      on='Sample File Name',
                      message='Blanks')

Re-generate the 'sample summary' report to check outcomes of exclusions:

In [ ]:
nPYc.reports.generateReport(msData, 'sample summary')

In [ ]:
data = nPYc.plotting.plotIntensityInteractive(msData,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True)
iplot(data)

Apply masks to permanently delete excluded samples:

In [ ]:
msData.applyMasks()

### Feature Summary Report

Generate the feature summary report, check the quality of the dataset with regards to quality control criteria.

In [ ]:
nPYc.reports.generateReport(msData, 'feature summary')

#### Export:

In [ ]:
nPYc.reports.generateReport(msData,
                            'sample summary',
                            destinationPath=qcDir)

nPYc.reports.generateReport(msData,
                            'feature summary', 
                            destinationPath=qcDir)

data = nPYc.plotting.plotIntensityInteractive(msData,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True,
                                              destinationPath=qcDir,
                                              autoOpen=False)

### OPTIONAL: Correlation to Dilution Report 

If correlation to dilution figures show sub-optimal results, for an interactive plot of dilution series samples run:

```
data = nPYc.plotting.plotTICinteractive(msData, plottype='Serial Dilution')
iplot(data)
```


Generate the correlation to dilution report for further information:

```
nPYc.reports.generateReport(msData, 'correlation to dilution')
```

# 6. Assess Batch and Run-Order Effects and Apply Correction if Necessary

Assess performance of batch and run-order correction on a subset of features:

In [ ]:
nPYc.reports.generateReport(msData,
                            'batch correction assessment',
                            withExclusions=True)

In [ ]:
msData.sampleMetadata.loc[msData.sampleMetadata['Correction Batch']==2]

To run the assessment report on just SS, SR and LTR samples:
    
```
msData.updateMasks(sampleTypes=[SampleType.StudySample, SampleType.StudyPool, SampleType.ExternalReference],
                   assayRoles=[AssayRole.Assay, AssayRole.PrecisionReference],
                   filterFeatures=False)

nPYc.reports.generateReport(msData,
                            'batch correction assessment',
                            withExclusions=True)

msData.initialiseMasks()
```

If data acquired in multiple batches, a new batch can be started using:

```
msData.amendBatches(x)
```

Where x is the run order of the first sample in the new batch, which can be determined using:

```
data = nPYc.plotting.plotTICinteractive(msData, plottype='Sample Type')
iplot(data)
```

### Run Batch Correction

In [ ]:
msData.sampleMetadata['Correction Batch'] = 1

In [ ]:
msDatacorrected = nPYc.batchAndROCorrection.correctMSdataset(msData,
                                                             parallelise=True)

### Batch Correction Summary Report

In [ ]:
nPYc.reports.generateReport(msData,
                            'batch correction summary',
                            msDataCorrected=msDatacorrected)

### Export:

In [ ]:
nPYc.reports.generateReport(msData,
                            'batch correction assessment',
                            withExclusions=True,
                            destinationPath=qcDirCorrected)

nPYc.reports.generateReport(msData,
                            'batch correction summary',
                            msDataCorrected=msDatacorrected,
                            destinationPath=qcDirCorrected)

# 7. Feature Filtering

For LC-MS datasets, features should be filtered based on their individual precision and accuracy to ensure that only high quality features are retained. 

The nPYc-Toolbox has been developed based on the quality control criteria previously described in [Lewis et al. 2016](https://www.ncbi.nlm.nih.gov/pubmed/27479709), and uses three key criteria for feature filtering, see [Sample and Feature Masks](https://npyc-toolbox.readthedocs.io/en/latest/masks.html):

1. Correlation to dilution (calculated from SRD samples), assess that features are measured accurately with respect to their true abundance
2. RSD (calculated from SR samples), assesses that features are measured precisely from multiple acquisitions of the same sample across the run
3. Comparing RSD in SR and SS, the variance in SS should always be greater than that seen in the SR samples

The distribution of correlation to dilution, and RSD can be visualised using the 'Feature Summary Report'.

A report summarising the number of features passing selection with different criteria can also be produced using:  

In [ ]:
# For uncorrected data
nPYc.reports.generateReport(msData, 'feature selection')

In [ ]:
# For corrected data
nPYc.reports.generateReport(msDatacorrected, 'feature selection')

If more (or less) stringent filtering criteria are required these can be set in the object 'Attributes', for example:

To amend the RSD threshold: 

```
msDatacorrected.Attributes['rsdThreshold'] = 20
```

Or to amend the correlation to dilution threshold:

```
msDatacorrected.Attributes['corrThreshold'] = .8
```

For more details and a full list of parameters see [Built-in Configuration SOPs](https://npyc-toolbox.readthedocs.io/en/latest/configuration/builtinSOPs.html).

Update the featureMask to mask those features not meeting feature filtering thresholds:

In [ ]:
# For uncorrected data
msData.updateMasks(filterSamples=False,
                   filterFeatures=True)

In [ ]:
# For corrected data
msDatacorrected.updateMasks(filterSamples=False,
                            filterFeatures=True)

Generate feature summary reports with feature filtering applied:

In [ ]:
# For uncorrected data
nPYc.reports.generateReport(msData,
                            'feature summary',
                            withExclusions=True)

In [ ]:
# For corrected data
nPYc.reports.generateReport(msDatacorrected,
                            'feature summary', 
                            withExclusions=True)

Interactive plots of the sum of feature intensities for all samples:

In [ ]:
# For uncorrected data
data = nPYc.plotting.plotIntensityInteractive(msData,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True)
iplot(data)

In [ ]:
# For corrected data
data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True)
iplot(data)

### Export:

In [ ]:
# For uncorrected data
nPYc.reports.generateReport(msData,
                            'feature selection',
                            destinationPath=qcDirUncorrected)

nPYc.reports.generateReport(msData,
                            'feature summary',
                            withExclusions=True,
                            destinationPath=qcDirUncorrected)

data = nPYc.plotting.plotIntensityInteractive(msData,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True,
                                              destinationPath=qcDirUncorrected,
                                              autoOpen=False)

In [ ]:
# For corrected data
nPYc.reports.generateReport(msDatacorrected, 
                            'feature selection', 
                            destinationPath=qcDirCorrected)

nPYc.reports.generateReport(msDatacorrected,
                            'feature summary', 
                            withExclusions=True, 
                            destinationPath=qcDirCorrected)

data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=True,
                                              destinationPath=qcDirCorrected,
                                              autoOpen=False)

Retain only SS, SR and LTR samples:

In [ ]:
# For uncorrected data
msData.updateMasks(sampleTypes=[SampleType.StudySample, SampleType.StudyPool, SampleType.ExternalReference],
                   assayRoles=[AssayRole.Assay, AssayRole.PrecisionReference],
                   filterFeatures=False)

In [ ]:
# For corrected data
msDatacorrected.updateMasks(sampleTypes=[SampleType.StudySample, SampleType.StudyPool, SampleType.ExternalReference],
                            assayRoles=[AssayRole.Assay, AssayRole.PrecisionReference],
                            filterFeatures=False)

# 8. Analytical Multivariate Quality Control

Generate a principal component analysis (PCA) model of the data use this to assess data quality, identify potential sample and feature outliers, and determine any potential analytical associations with the main sources of variance in the data.

One key parameter is 'scaling', recommended values are 0 for mean-centering only (appropriate for NMR data), 0.5 for Pareto scaling and 1 for unit variance (UV) scaling (appropriate for LC-MS data).

Generate PCA model and reports:

In [ ]:
PCAmodelAnalytical = nPYc.multivariate.exploratoryAnalysisPCA(msData,
                                                              withExclusions=True)

In [ ]:
# For uncorrected data


nPYc.reports.multivariateReport(msData,
                                PCAmodelAnalytical,
                                reportType='analytical',
                                withExclusions=True)

In [ ]:
# For corrected data
PCAmodelAnalyticalcorrected = nPYc.multivariate.exploratoryAnalysisPCA(msDatacorrected,
                                                                       withExclusions=True)

nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelAnalyticalcorrected,
                                reportType='analytical',
                                withExclusions=True)

Scores and loadings plots can also be explored interactively with the 'plotScoresInteractive' and 'plotLoadingsInteractive' functions.

**Interactive scores plot**

For example, to plot the scores plot for principal component 1 vs. principal component  2 ("components=[1, 2]") with points coloured by values in msDatacorrected.sampleMetadata['Run Order'] (the colour is defined by the third input argument and can be any column name in the sample metadata):

In [ ]:
data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                           PCAmodelAnalyticalcorrected,
                                           colourBy='Run Order', 
                                           labelBy='Run Order',
                                           components=[1, 2], 
                                           withExclusions=True)
iplot(data)

**Interactive loadings plot**

For example, to plot the loadings plot for principal component 1:

In [ ]:
data = nPYc.plotting.plotLoadingsInteractive(msDatacorrected,
                                             PCAmodelAnalyticalcorrected,
                                             component=1,
                                             withExclusions=True)
iplot(data)

### Export:

In [ ]:
# For uncorrected data
nPYc.reports.multivariateReport(msData,
                                PCAmodelAnalytical,
                                reportType='analytical',
                                withExclusions=True,
                                destinationPath=qcDirUncorrected)

# Export scores and loadings (values and interactive plots)
loadings = copy.deepcopy(msData.featureMetadata.loc[msData.featureMask,:])
scores = copy.deepcopy(msData.sampleMetadata.loc[msData.sampleMask,:])

for i in numpy.arange(PCAmodelAnalytical.loadings.shape[0]):

    c1 = int(i+1)
    c2 = int(i+2)

    # Plot scores (in pairs, so only plot when c1 is odd, set c2 to 1 if odd number of total components)
    if c1 % 2 != 0:

        if c2 > PCAmodelAnalytical.loadings.shape[0]: c2 = 1
        
        data = nPYc.plotting.plotScoresInteractive(msData,
                                                   PCAmodelAnalytical,
                                                   colourBy='Run Order',
                                                   labelBy='Run Order',
                                                   components=[c1, c2],
                                                   withExclusions=True,
                                                   destinationPath=qcDirUncorrected,
                                                   autoOpen=False)

    # Plot loadings
    data = nPYc.plotting.plotLoadingsInteractive(msData,
                                                 PCAmodelAnalytical,
                                                 component=c1,
                                                 withExclusions=True,
                                                 destinationPath=qcDirUncorrected,
                                                 autoOpen=False)

    # Save values
    loadings['PC'+str(i+1)] = PCAmodelAnalytical.loadings[i,:]
    scores['PC'+str(i+1)] = PCAmodelAnalytical.scores[:,i]

# Export
loadings.to_csv(os.path.join(qcDirUncorrected, msData.name + '_featureMetadata_plus_loadings.csv'))
scores.to_csv(os.path.join(qcDirUncorrected, msData.name + '_sampleMetadata_plus_scores.csv'))

In [ ]:
# For corrected data
nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelAnalyticalcorrected,
                                reportType='analytical',
                                withExclusions=True,
                                destinationPath=qcDirCorrected)

# Export scores and loadings (values and interactive plots)
loadingscorrected = copy.deepcopy(msDatacorrected.featureMetadata.loc[msDatacorrected.featureMask,:])
scorescorrected = copy.deepcopy(msDatacorrected.sampleMetadata.loc[msDatacorrected.sampleMask,:])

for i in numpy.arange(PCAmodelAnalyticalcorrected.loadings.shape[0]):

    c1 = int(i+1)
    c2 = int(i+2)

    # Plot scores (in pairs, so only plot when c1 is odd, set c2 to 1 if odd number of total components)
    if c1 % 2 != 0:

        if c2 > PCAmodelAnalyticalcorrected.loadings.shape[0]: c2 = 1
        
        data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                                   PCAmodelAnalyticalcorrected,
                                                   colourBy='Run Order',
                                                   labelBy='Run Order',
                                                   components=[c1, c2],
                                                   withExclusions=True,
                                                   destinationPath=qcDirCorrected,
                                                   autoOpen=False)

    # Plot loadings
    data = nPYc.plotting.plotLoadingsInteractive(msDatacorrected,
                                                 PCAmodelAnalyticalcorrected,
                                                 component=c1,
                                                 withExclusions=True,
                                                 destinationPath=qcDirCorrected,
                                                 autoOpen=False)

    # Save values
    loadingscorrected['PC'+str(i+1)] = PCAmodelAnalyticalcorrected.loadings[i,:]
    scorescorrected['PC'+str(i+1)] = PCAmodelAnalyticalcorrected.scores[:,i]

loadingscorrected.to_csv(os.path.join(qcDirCorrected, msDatacorrected.name + '_featureMetadata_plus_loadings.csv'))
scorescorrected.to_csv(os.path.join(qcDirCorrected, msDatacorrected.name + '_sampleMetadata_plus_scores.csv'))

Generate final report and export:

In [ ]:
# For uncorrected data
nPYc.reports.generateReport(msData,
                            'final report',
                            withExclusions=True)

nPYc.reports.generateReport(msData,
                            'final report',
                            withExclusions=True,
                            destinationPath=qcDirUncorrected)

In [ ]:
# For corrected data
nPYc.reports.generateReport(msDatacorrected,
                            'final report',
                            withExclusions=True)

nPYc.reports.generateReport(msDatacorrected,
                            'final report',
                            withExclusions=True,
                            destinationPath=qcDirCorrected)

# 9. Finalise and Export Dataset

Apply final feature and sample filtering.

NOTE: all following steps should be applied to one of corrected OR non-corrected data objects depending on the final decision for batch correction.

Create QC directory for FINAL data:

In [ ]:
qcDirFinal = os.path.join(qcDir, 'Final')

In [ ]:
# If batch correction NOT to be applied replace msDatacorrected with msData
#msDatacorrected = msData

Apply masks to permanently exclude features:

In [ ]:
msDatacorrected.applyMasks()

Retain only samples required (default is SS and SR only):

In [ ]:
msDatacorrected.updateMasks(sampleTypes=[SampleType.StudySample, SampleType.StudyPool],
                            assayRoles=[AssayRole.Assay, AssayRole.PrecisionReference],
                            filterFeatures=False)

Include PQN factors in MV report to check scores against PQN derived sample concentrations:

In [ ]:
msDatapqn = copy.deepcopy(msDatacorrected)

In [ ]:
msDatapqn.Normalisation = ProbabilisticQuotientNormaliser()
msDatapqn.intensityData

In [ ]:
msDatacorrected.sampleMetadata['PQN factors'] = msDatapqn.Normalisation.normalisation_coefficients

In [ ]:
nPYc.plotting.histogram(msDatacorrected.sampleMetadata['PQN factors'])

In [ ]:
msDatacorrected.Attributes['analyticalMeasurements']['PQN factors'] = 'continuous'

Generate and export multivariate report:

In [ ]:
PCAmodelAnalyticalcorrected = nPYc.multivariate.exploratoryAnalysisPCA(msDatacorrected,
                                                                       withExclusions=True)

In [ ]:
nPYc.reports.multivariateReport(msDatacorrected, 
                                PCAmodelAnalyticalcorrected, 
                                reportType='analytical', 
                                withExclusions=True)

In [ ]:
msDatacorrected.sampleMetadata.columns

In [ ]:
data = nPYc.plotting.plotScoresInteractive(msDatacorrected, 
                                           PCAmodelAnalyticalcorrected, 
                                           colourBy='Sample ID',
                                           labelBy='Sample ID',
                                           components=[1, 2], 
                                           withExclusions=True)
iplot(data)

In [ ]:
data = nPYc.plotting.plotLoadingsInteractive(msDatacorrected, 
                                             PCAmodelAnalyticalcorrected, 
                                             component=1, 
                                             withExclusions=True)
iplot(data)

Export (NOTE - make sure to include scores plotted by any factor of interest from the report):

In [ ]:
# For corrected data
nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelAnalyticalcorrected,
                                reportType='analytical',
                                withExclusions=True,
                                destinationPath=qcDirFinal)

# Export scores and loadings (values and interactive plots)
loadingscorrected = copy.deepcopy(msDatacorrected.featureMetadata.loc[msDatacorrected.featureMask,:])
scorescorrected = copy.deepcopy(msDatacorrected.sampleMetadata.loc[msDatacorrected.sampleMask,:])

for i in numpy.arange(PCAmodelAnalyticalcorrected.loadings.shape[0]):

    c1 = int(i+1)
    c2 = int(i+2)

    # Plot scores (in pairs, so only plot when c1 is odd, set c2 to 1 if odd number of total components)
    if c1 % 2 != 0:

        if c2 > PCAmodelAnalyticalcorrected.loadings.shape[0]: c2 = 1
        
        data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                                   PCAmodelAnalyticalcorrected,
                                                   colourBy='Run Order',
                                                   labelBy='Run Order',
                                                   components=[c1, c2],
                                                   withExclusions=True,
                                                   destinationPath=qcDirFinal,
                                                   autoOpen=False)

        data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                                   PCAmodelAnalyticalcorrected,
                                                   colourBy='Sample ID',
                                                   labelBy='Sample ID',
                                                   components=[c1, c2],
                                                   withExclusions=True,
                                                   destinationPath=qcDirFinal,
                                                   autoOpen=False)
    
    # Plot loadings
    data = nPYc.plotting.plotLoadingsInteractive(msDatacorrected,
                                                 PCAmodelAnalyticalcorrected,
                                                 component=c1,
                                                 withExclusions=True,
                                                 destinationPath=qcDirFinal,
                                                 autoOpen=False)

    # Save values
    loadingscorrected['PC'+str(i+1)] = PCAmodelAnalyticalcorrected.loadings[i,:]
    scorescorrected['PC'+str(i+1)] = PCAmodelAnalyticalcorrected.scores[:,i]

loadingscorrected.to_csv(os.path.join(qcDirFinal, msDatacorrected.name + '_featureMetadata_plus_loadings.csv'))
scorescorrected.to_csv(os.path.join(qcDirFinal, msDatacorrected.name + '_sampleMetadata_plus_scores.csv'))

Generate the final feature report and total intensity plot and export:

In [ ]:
data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='Run Order',
                                              labelBy='Run Order',
                                              withExclusions=True)
iplot(data)

In [ ]:
nPYc.reports.generateReport(msDatacorrected,
                            'feature summary')

data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='Sample ID',
                                              labelBy='Sample ID',
                                              withExclusions=True)
iplot(data)

In [ ]:
nPYc.reports.generateReport(msDatacorrected,
                            'feature summary', 
                            destinationPath=qcDirFinal)

data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='Sample ID',
                                              labelBy='Sample ID',
                                              withExclusions=True,
                                              destinationPath=qcDirFinal,
                                              autoOpen=False)

Generate the final report and export:

In [ ]:
nPYc.reports.generateReport(msDatacorrected,
                            'final report')

In [ ]:
nPYc.reports.generateReport(msDatacorrected,
                            'final report',
                            destinationPath=qcDirFinal)

# 10. Review by LC-MS Manager/Team

At this stage reports should be reviewed by the team and final decisions taken on the following:

- Sample exclusions
- Batch and run-order correction
- Feature filtering criteria
- Any other issues

Once these have been decided the final dataset can be generated with these parameters.

# 11. Export final dataset

NOTE: if you are happy with the data quality and pre-processing, the final dataset and report could be generated at this stage, but only shared with collaborator after review by the MS manager/team.

In [ ]:
msDatacorrected.applyMasks()

In [ ]:
# Standard format
msDatacorrected.exportDataset(destinationPath=saveDir)

# Combined format
msDatacorrected.exportDataset(destinationPath=saveDir,
                              saveFormat='UnifiedCSV')

# Report
nPYc.reports.generateReport(msDatacorrected,
                            'final report',
                            destinationPath=saveDir)

# 12. OPTIONAL: Normalise Data and Export

Dilution effects on global sample intensity can be normalised by using one of the available [Normalisation classes](https://npyc-toolbox.readthedocs.io/en/latest/normalisation.html).

This is especially critical in urine samples when samples can vary substantially in concentration. Here we apply Probabilistic Quotient Normalisation (PQN) [Deterle et al. 2006](https://pubs.acs.org/doi/10.1021/ac051632c).

Setting the 'Normalisation' parameter in the dataset object automatically applies the selected normalisation.

First change name to indicate data has been normalised and create directory to save outputs:

In [ ]:
msDatacorrected.name = msDatacorrected.name + '_PQN_normalised'

qcDirPQN = os.path.join(qcDir, 'PQN')

Apply required normalisation:

In [ ]:
msDatacorrected.Normalisation = ProbabilisticQuotientNormaliser()
msDatacorrected.intensityData

Extract coefficients and plot:

In [ ]:
PQNfactors = msDatacorrected.Normalisation.normalisation_coefficients

msDatacorrected.sampleMetadata['PQN factors'] = msDatacorrected.Normalisation.normalisation_coefficients

PQNtable = pandas.DataFrame(numpy.c_[msDatacorrected.sampleMetadata['Sample File Name'],
                                     msDatacorrected.sampleMetadata['Sample ID'],
                                     PQNfactors],
                            columns=['Sample File Name',
                                     'Sample ID',
                                     'PQN factors'])

nPYc.plotting.histogram(PQNfactors)

Check normalised data - final and multivariate reports:

In [ ]:
nPYc.reports.generateReport(msDatacorrected,
                            'final report')

In [ ]:
data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='Plate',
                                              labelBy='Plate',
                                              withExclusions=False)
iplot(data)

In [ ]:
PCAmodelAnalytical = nPYc.multivariate.exploratoryAnalysisPCA(msDatacorrected,
                                                              withExclusions=True)

nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelAnalytical,
                                reportType='analytical',
                                withExclusions=True)

In [ ]:
data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                           PCAmodelAnalytical,
                                           colourBy='Plate',
                                           labelBy='Plate', 
                                           components=[1, 2], 
                                           withExclusions=True)
iplot(data)

#### Export

In [ ]:
# Export reports
nPYc.reports.generateReport(msDatacorrected,
                            'final report',
                            destinationPath=qcDirPQN)

data = nPYc.plotting.plotIntensityInteractive(msDatacorrected,
                                              x='Acquired Time',
                                              y='Sum of Feature Intensities',
                                              colourBy='SampleClass',
                                              labelBy='SampleClass',
                                              withExclusions=False,
                                              destinationPath=qcDirPQN,
                                              autoOpen=False)

nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelAnalytical,
                                reportType='analytical',
                                withExclusions=True,
                                destinationPath=qcDirPQN)

# Export scores and loadings (values and interactive plots)
loadingscorrected = copy.deepcopy(msDatacorrected.featureMetadata.loc[msDatacorrected.featureMask,:])
scorescorrected = copy.deepcopy(msDatacorrected.sampleMetadata.loc[msDatacorrected.sampleMask,:])

for i in numpy.arange(PCAmodelAnalytical.loadings.shape[0]):

    c1 = int(i+1)
    c2 = int(i+2)

    # Plot scores (in pairs, so only plot when c1 is odd, set c2 to 1 if odd number of total components)
    if c1 % 2 != 0:

        if c2 > PCAmodelAnalytical.loadings.shape[0]: c2 = 1
        
        data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                                   PCAmodelAnalytical,
                                                   colourBy='Run Order',
                                                   labelBy='Run Order',
                                                   components=[c1, c2],
                                                   withExclusions=True,
                                                   destinationPath=qcDirPQN,
                                                   autoOpen=False)

        data = nPYc.plotting.plotScoresInteractive(msDatacorrected,
                                                   PCAmodelAnalytical,
                                                   colourBy='Sample ID',
                                                   labelBy='Sample ID',
                                                   components=[c1, c2],
                                                   withExclusions=True,
                                                   destinationPath=qcDirPQN,
                                                   autoOpen=False)
    
    # Plot loadings
    data = nPYc.plotting.plotLoadingsInteractive(msDatacorrected,
                                                 PCAmodelAnalytical,
                                                 component=c1,
                                                 withExclusions=True,
                                                 destinationPath=qcDirPQN,
                                                 autoOpen=False)

    # Save values
    loadingscorrected['PC'+str(i+1)] = PCAmodelAnalytical.loadings[i,:]
    scorescorrected['PC'+str(i+1)] = PCAmodelAnalytical.scores[:,i]

loadingscorrected.to_csv(os.path.join(qcDirPQN, msDatacorrected.name + '_featureMetadata_plus_loadings.csv'))
scorescorrected.to_csv(os.path.join(qcDirPQN, msDatacorrected.name + '_sampleMetadata_plus_scores.csv'))

In [ ]:
# Export PQN coefficients to csv
PQNtable.to_csv(os.path.join(qcDirPQN,
                             msDatacorrected.name + ' PQN Normalisation factors.csv'),
                index=False)

In [ ]:
# Export final dataset (standard format, three file output)
msDatacorrected.exportDataset(destinationPath=qcDirPQN)

# Export final summary report
nPYc.reports.generateReport(msDatacorrected,
                            'final report',
                            destinationPath=qcDirPQN)

# Export combined dataset (e.g., format for SIMCA)
msDatacorrected.exportDataset(destinationPath=qcDirPQN,
                              saveFormat='UnifiedCSV')

# 12. OPTIONAL: Biological Multivariate Report

The final optional step is to generate the biological equivalent of the multivariate report.

By default, all non-analytical parameters are used, but if required specific fields only can be reported by defining a dictionary of fields required (and their types) and entering this as an additional input parameter to the reporting:

```
# For example, using 'Subject ID' and 'Age' (these must be fields in the sampleMetadata)
biologicalMeasurements = {'Subject ID': 'categorical',
                          'Age': 'continuous'}
```

Keep study samples only:

In [ ]:
msDatacorrected.updateMasks(sampleTypes=[SampleType.StudySample],
                            filterFeatures=False)

Generate the PCA model and multivariate report:

In [ ]:
PCAmodelBiological = nPYc.multivariate.exploratoryAnalysisPCA(msDatacorrected,
                                                              withExclusions=True,
                                                              scaling='uv')

In [ ]:
nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelBiological,
                                reportType='biological',
                                #biologicalMeasurements=biologicalMeasurements,
                                withExclusions=True)

#### Export:

In [ ]:
nPYc.reports.multivariateReport(msDatacorrected,
                                PCAmodelBiological,
                                reportType='biological',
                                #biologicalMeasurements=biologicalMeasurements,
                                withExclusions=True,
                                destinationPath=saveDir)